In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable
from pyspark.sql.types import *


In [0]:
initial_path = "/Volumes/final_project/source/raw_data/loan_repayment_initial.csv"

print("Initial file:", initial_path)
 

In [0]:
display(dbutils.fs.ls("/Volumes/final_project/source/raw_data/loan_repayment_initial.csv"))

In [0]:
# Read initial loan repayment data

source_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(initial_path)
)

display(source_df)

In [0]:
df.printSchema()

In [0]:
# Display column names
print("Columns:", source_df.columns)

In [0]:
# Display source record count
print("Source Count:", source_df.count())

In [0]:
print("Total source records:", df.count())

In [0]:
# Display first 10 records
display(source_df.limit(10))

In [0]:

# Add audit columns to initial source data

bronze_raw_df = (
    source_df
    .withColumn("source_file", lit("loan_repayment_initial.csv"))
    .withColumn("batch_id", lit("BATCH_001"))
    .withColumn("ingestion_timestamp", current_timestamp())
)

display(bronze_raw_df)

In [0]:
bronze_raw_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("Final_project.bronze.loan_repayment_raw")

print("Raw Bronze table created successfully")

In [0]:
from pyspark.sql.functions import *

df.select([
    count(when(col(c).isNull(), 1)).alias(c)
    for c in df.columns
]).display()

In [0]:
from pyspark.sql.window import Window

# Check duplicate payment IDs
payment_window = Window.partitionBy("payment_id")

validated_df = (
    bronze_raw_df
    .withColumn("duplicate_count", count("*").over(payment_window))
    .withColumn(
        "rejection_reason",
        concat_ws(
            ", ",
            when(
                col("payment_id").isNull() | (trim(col("payment_id")) == ""),
                lit("Missing payment_id")
            ),
            when(
                col("loan_id").isNull() | (trim(col("loan_id")) == ""),
                lit("Missing loan_id")
            ),
            when(
                col("customer_id").isNull() | (trim(col("customer_id")) == ""),
                lit("Missing customer_id")
            ),
            when(
                col("principal_amount").isNull() |
                (col("principal_amount") < 0),
                lit("Invalid principal_amount")
            ),
            when(
                col("interest_amount").isNull() |
                (col("interest_amount") < 0),
                lit("Invalid interest_amount")
            ),
            when(
                col("penalty_amount").isNull() |
                (col("penalty_amount") < 0),
                lit("Invalid penalty_amount")
            ),
            when(
                (coalesce(col("principal_amount"), lit(0)) <= 0) &
                (coalesce(col("interest_amount"), lit(0)) <= 0) &
                (coalesce(col("penalty_amount"), lit(0)) <= 0),
                lit("All amount components are zero")
            ),
            when(
                to_date(col("payment_date")).isNull(),
                lit("Invalid payment_date")
            ),
            when(
                col("loan_type").isNull() |
                (~upper(trim(col("loan_type"))).isin("HOME", "VEHICLE", "PERSONAL")),
                lit("Invalid loan_type")
            ),
            when(
                col("payment_status").isNull() |
                (~upper(trim(col("payment_status"))).isin("PENDING", "PAID", "FAILED")),
                lit("Invalid payment_status")
            ),
            when(
                col("duplicate_count") > 1,
                lit("Duplicate payment_id")
            )
        )
    )
)

display(validated_df)

In [0]:
# Separate valid and rejected records

valid_bronze_df = validated_df.filter(col("rejection_reason") == "")

rejected_bronze_df = validated_df.filter(col("rejection_reason") != "")


In [0]:
# Display counts
print("Valid Records:", valid_bronze_df.count())
print("Rejected Records:", rejected_bronze_df.count())

In [0]:
# Display valid records
display(valid_bronze_df)

In [0]:
# Display rejected records with reasons
display(
    rejected_bronze_df.select(
        "payment_id",
        "loan_id",
        "customer_id",
        "rejection_reason"
    )
)

In [0]:
# Save valid Bronze records
valid_bronze_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("Final_project.bronze.loan_repayment_valid")

In [0]:
# Save rejected Bronze records
rejected_bronze_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("Final_project.bronze.loan_repayment_rejected")

print("Valid and rejected Bronze tables saved successfully")

In [0]:
# Get record counts

source_count = source_df.count()
valid_count = valid_bronze_df.count()
rejected_count = rejected_bronze_df.count()

In [0]:
# Reconciliation calculation
print("Source Count:", source_count)
print("Valid Count:", valid_count)
print("Rejected Count:", rejected_count)

if source_count == valid_count + rejected_count:
    print("Reconciliation Status: PASS")
else:
    print("Reconciliation Status: FAIL")